In [1]:
import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import t as t_dist

# ---------------------------
# 1. Define cryptocurrencies and their types
# ---------------------------
crypto_types = {
    "Bitcoin": "brown",            # Bitcoin (originally BTC)
    "Cardano": "green",            # Cardano (ADA)
    "Cosmos": "green",             # Cosmos (ATOM)
    "Dogecoin": "brown",           # Dogecoin
    "Ethereum Classic": "brown",   # Ethereum Classic (ETC)
    "Ethereum": "brown",           # Ethereum (ETH)
    "Litecoin": "brown",           # Litecoin
    "Monero": "brown",             # Monero
    "Polygon": "green",            # Polygon (MATIC)
    "Solana": "green",             # Solana
    "Stellar": "green",            # Stellar (XLM)
    "Tron": "green",               # Tron (TRX)
    "Ripple": "green"              # Ripple (XRP)
}

# ---------------------------
# 2. Set the directory path for the data files
# ---------------------------
directory = r"D:\桌面\绿色货币\数据总库\加密货币数据\coinmarketcap\weekly"  # Do not change this path

# ---------------------------
# 3. Loop through the directory to read CSV files and merge the data
# ---------------------------
dfs = []
selected_columns = ['name', 'open', 'high', 'low', 'close', 'volume', 'marketCap', 'timestamp']

for f in os.listdir(directory):
    ticker = f.split("_")[0]
    if f.endswith(".csv") and ticker in crypto_types:
        file_path = os.path.join(directory, f)
        try:
            df = pd.read_csv(file_path, sep=";", quotechar='"', parse_dates=["timestamp"])
            df.drop(columns=["Start"], errors="ignore", inplace=True)
            df = df[selected_columns]
            df["source"] = ticker
            df["Type"] = crypto_types[ticker]
            dfs.append(df)
        except Exception as e:
            print(f"Error reading {f}: {e}")

data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
data['timestamp'] = pd.to_datetime(data['timestamp']).dt.strftime('%Y-%m-%d')
data.rename(columns={"timestamp": "Date"}, inplace=True)

# ---------------------------
# 4. Data Preprocessing
# ---------------------------
df = data.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df[df["Date"] >= "2019-06-30"]  # Filter by date range as needed
df = df.sort_values(["source", "Date"]).reset_index(drop=True)

# ---------------------------
# 5. Calculate log returns (rtn) and next period log returns (rtn_next)
# ---------------------------
df["rtn"] = df.groupby("source")["close"].transform(lambda x: np.log(x).diff())
df["rtn_next"] = df.groupby("source")["rtn"].shift(-1)
df = df.dropna()

# ---------------------------
# 6. Factor Construction
# ---------------------------

# 6.1 Market Factor (CMKT)
# For each date, compute the market return as the value-weighted log return of all coins,
# where the weight is the current market capitalization.
CMKT = df.groupby("Date").apply(lambda g: np.sum(g["rtn"] * g["marketCap"]) / np.sum(g["marketCap"]))
CMKT.name = "CMKT"

# 6.2 Size Factor (CSMB)
# For each date, split the coins into three groups by market capitalization:
# Small (bottom 30%), Middle (middle 40%), and Big (top 30%).
df_size = df.groupby("Date", group_keys=False).apply(
    lambda sub: sub.assign(Size_Group=pd.qcut(sub["marketCap"], [0, 0.3, 0.7, 1.0], labels=["Small", "Middle", "Big"]))
).reset_index(drop=True)

# Calculate value-weighted average log returns for Small and Big groups.
small_vw_return = df_size[df_size["Size_Group"] == "Small"].groupby("Date").apply(
    lambda g: np.sum(g["rtn"] * g["marketCap"]) / np.sum(g["marketCap"])
)
big_vw_return = df_size[df_size["Size_Group"] == "Big"].groupby("Date").apply(
    lambda g: np.sum(g["rtn"] * g["marketCap"]) / np.sum(g["marketCap"])
)
# The size factor is the difference between the Small and Big portfolios' returns.
CSMB = small_vw_return - big_vw_return
CSMB.name = "CSMB"

# 6.3 Momentum Factor (CMOM) using 3-period momentum
# First, compute each coin's cumulative log return over the past 3 periods.
df_size["momentum"] = df_size.groupby("source")["rtn"].transform(
    lambda x: (1 + x).rolling(window=3, min_periods=3).apply(np.prod, raw=True) - 1)
df_size = df_size.dropna()

# Only consider the Small and Big groups.
df_mom = df_size[df_size["Size_Group"].isin(["Small", "Big"])].copy()

# For each date and each size group, sort by the 3-period cumulative return and split into Low, Middle, and High groups.
def assign_momentum_group_sorted(sub):
    sub = sub.sort_values("momentum").reset_index(drop=True)
    n = len(sub)
    idx_low = int(np.floor(n * 0.3))
    idx_high = int(np.floor(n * 0.7))
    sub.loc[sub.index[:idx_low], "Momentum_Group"] = "Low"
    sub.loc[sub.index[idx_low:idx_high], "Momentum_Group"] = "Middle"
    sub.loc[sub.index[idx_high:], "Momentum_Group"] = "High"
    return sub

df_mom = df_mom.groupby(["Date", "Size_Group"], group_keys=False).apply(assign_momentum_group_sorted)
df_mom = df_mom.reset_index(drop=True)

# Construct the momentum factor using the formula:
# CMOM = 0.5 * (Return(Small High) + Return(Big High)) - 0.5 * (Return(Small Low) + Return(Big Low))
# Here, we compute value-weighted returns.
def compute_cmom(group):
    # Extract subgroups.
    small_high = group[(group["Size_Group"]=="Small") & (group["Momentum_Group"]=="High")]
    big_high   = group[(group["Size_Group"]=="Big")   & (group["Momentum_Group"]=="High")]
    small_low  = group[(group["Size_Group"]=="Small") & (group["Momentum_Group"]=="Low")]
    big_low    = group[(group["Size_Group"]=="Big")   & (group["Momentum_Group"]=="Low")]
    
    # If any subgroup is empty, return NaN.
    if small_high.empty or big_high.empty or small_low.empty or big_low.empty:
        return np.nan

    # Calculate the value-weighted average returns for each subgroup.
    small_high_w = np.sum(small_high["rtn"] * small_high["marketCap"]) / np.sum(small_high["marketCap"])
    big_high_w   = np.sum(big_high["rtn"] * big_high["marketCap"]) / np.sum(big_high["marketCap"])
    small_low_w  = np.sum(small_low["rtn"] * small_low["marketCap"]) / np.sum(small_low["marketCap"])
    big_low_w    = np.sum(big_low["rtn"] * big_low["marketCap"]) / np.sum(big_low["marketCap"])
    
    # Construct the momentum factor.
    return 0.5 * (small_high_w + big_high_w) - 0.5 * (small_low_w + big_low_w)

CMOM = df_mom.groupby("Date").apply(compute_cmom)
CMOM.name = "CMOM"

print("Preview of CMKT factor:")
print(CMKT.head())
print("Preview of CSMB factor:")
print(CSMB.head())
print("Preview of CMOM factor:")
print(CMOM.head())

# ---------------------------
# 7. Read risk-free rate data and compute the value-weighted next period excess return
# ---------------------------
rf_file_path = r"D:\桌面\绿色货币\数据总库\宏观经济数据\美国国债\us_treasury_daliy_1M.csv"  # Do not change this path
rf_data = pd.read_csv(rf_file_path)
rf_data["Date"] = pd.to_datetime(rf_data["Date"])
rf_data["Rf"] = rf_data["1 Mo"] / 100  # Convert "1 Mo" from percentage to decimal
rf_data = rf_data[["Date", "Rf"]]
rf_data = rf_data.set_index("Date").resample("W").mean().reset_index()

# Merge risk-free rate data into df (current period)
df = df.merge(rf_data, on="Date", how="left")
# For each coin, get the next period's risk-free rate using shift(-1)
df["Rf_next"] = df.groupby("source")["Rf"].shift(-1)
# Compute each coin's next period excess return: rp = rtn_next - Rf_next
df["rp"] = df["rtn_next"] - df["Rf_next"]

# For each Date and Type (e.g., green, brown), compute the value-weighted next period excess return (rp_next)
df_port = df.groupby(["Date", "Type"]).apply(
    lambda g: np.sum(g["rp"] * g["marketCap"]) / np.sum(g["marketCap"])
).reset_index().rename(columns={0: "rp_next"})
df_port = df_port.dropna()
print("Preview of value-weighted excess returns (rp_next):")
print(df_port.head())

# ---------------------------
# 8. Construct Factor DataFrame (using factors at current time t)
# ---------------------------
# Merge the CMKT, CSMB, and CMOM factors
test_factors = pd.concat([CMKT, CSMB, CMOM], axis=1).dropna().reset_index()

# Adjust the market factor to form an excess market factor using the current risk-free rate.
# Here, we use the risk-free rate corresponding to the day before the current date.
test_factors["prev_date"] = test_factors["Date"] - pd.Timedelta(days=1)
test_factors = pd.merge_asof(
    test_factors.sort_values("prev_date"),
    rf_data.sort_values("Date"),
    left_on="prev_date",
    right_on="Date",
    direction="backward"
)
test_factors.drop(columns=["prev_date", "Date_y"], inplace=True)
test_factors.rename(columns={"Date_x": "Date"}, inplace=True)
# Excess market factor: current CMKT minus the corresponding risk-free rate Rf
test_factors["CMKT"] = test_factors["CMKT"] - test_factors["Rf"]

# ---------------------------
# 9. Regression Analysis
# ---------------------------
# Merge the factor data with the value-weighted next period excess return data
df_reg = pd.merge(df_port, test_factors, on="Date", how="inner")

# For different types of cryptocurrencies (e.g., green vs. brown), run regressions with rp_next as the dependent variable.
green_model = smf.ols("rp_next ~ CMKT + CSMB + CMOM", data=df_reg[df_reg["Type"] == "green"]).fit()
brown_model = smf.ols("rp_next ~ CMKT + CSMB + CMOM", data=df_reg[df_reg["Type"] == "brown"]).fit()

print(green_model.summary())
print(brown_model.summary())

alpha_green = green_model.params["Intercept"]
se_green = green_model.bse["Intercept"]
alpha_brown = brown_model.params["Intercept"]
se_brown = brown_model.bse["Intercept"]

t_stat = (alpha_green - alpha_brown) / np.sqrt(se_green**2 + se_brown**2)
df_eff = min(green_model.df_resid, brown_model.df_resid)
p_val = 2 * (1 - t_dist.cdf(abs(t_stat), df=df_eff))

print("\nAlpha difference test (Green vs. Brown):")
print(f"Green alpha: {alpha_green:.4f} (SE={se_green:.4f})")
print(f"Brown alpha: {alpha_brown:.4f} (SE={se_brown:.4f})")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.4f}")


C:\Users\why97\AppData\Local\Temp\ipykernel_18000\3637604867.py:77: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CMKT = df.groupby("Date").apply(lambda g: np.sum(g["rtn"] * g["marketCap"]) / np.sum(g["marketCap"]))
C:\Users\why97\AppData\Local\Temp\ipykernel_18000\3637604867.py:83: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_size = df.groupby("Date", group_keys=False).apply(
C:\Users\why97\AppData\Loc

Preview of CMKT factor:
Date
2019-07-07    0.050633
2019-07-14   -0.147687
2019-07-21    0.034321
2019-07-28   -0.097442
2019-08-04    0.117301
Name: CMKT, dtype: float64
Preview of CSMB factor:
Date
2019-07-07   -0.024853
2019-07-14   -0.164148
2019-07-21    0.049073
2019-07-28   -0.004422
2019-08-04   -0.103706
Name: CSMB, dtype: float64
Preview of CMOM factor:
Date
2019-07-21    0.090207
2019-07-28    0.028968
2019-08-04    0.123601
2019-08-11    0.066842
2019-08-18    0.076589
Name: CMOM, dtype: float64
Preview of value-weighted excess returns (rp_next):
        Date   Type   rp_next
0 2019-07-07  brown -0.163867
1 2019-07-07  green -0.288614
2 2019-07-14  brown  0.008506
3 2019-07-14  green  0.057036
4 2019-07-21  brown -0.120593
                            OLS Regression Results                            
Dep. Variable:                rp_next   R-squared:                       0.044
Model:                            OLS   Adj. R-squared:                  0.033
Method:           

C:\Users\why97\AppData\Local\Temp\ipykernel_18000\3637604867.py:144: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CMOM = df_mom.groupby("Date").apply(compute_cmom)
C:\Users\why97\AppData\Local\Temp\ipykernel_18000\3637604867.py:172: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_port = df.groupby(["Date", "Type"]).apply(
